# Unit 2.1 The smolagents framework

In [1]:
# Activa un ejercicio externo únicamente después de preparar sus requisitos.
# Las opciones se pueden combinar si una práctica depende de otra.
external_exercises = set()

def install_packages(*args):
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)


### Menú como herramienta de Python



In [2]:
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion (str): The type of occasion for the party. Allowed values are:
                        - "casual": Menu for casual party.
                        - "formal": Menu for formal party.
                        - "superhero": Menu for superhero party.
                        - "custom": Custom menu.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

for occasion in ("casual", "formal", "superhero", "custom"):
    print(f"{occasion}: {suggest_menu(occasion)}")
assert suggest_menu("formal") == "3-course dinner with wine and dessert."


casual: Pizza, snacks, and drinks.
formal: 3-course dinner with wine and dessert.
superhero: Buffet with high-energy and healthy food.
custom: Custom menu for the butler.


In [3]:
if "code_agents" in external_exercises:
    install_packages(*['smolagents', '-U'])


In [4]:
if "code_agents" in external_exercises:
    from huggingface_hub import login

    login()


In [5]:
if "code_agents" in external_exercises:
    from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

    agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=InferenceClientModel())

    agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")


```python
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  results = web_search(query="best music for a Batman party")                                                      
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
```

In [6]:
if "code_agents" in external_exercises:
    from smolagents import CodeAgent, tool, InferenceClientModel

    # Tool to suggest a menu based on the occasion
    @tool
    def suggest_menu(occasion: str) -> str:
        """
        Suggests a menu based on the occasion.
        Args:
            occasion (str): The type of occasion for the party. Allowed values are:
                            - "casual": Menu for casual party.
                            - "formal": Menu for formal party.
                            - "superhero": Menu for superhero party.
                            - "custom": Custom menu.
        """
        if occasion == "casual":
            return "Pizza, snacks, and drinks."
        elif occasion == "formal":
            return "3-course dinner with wine and dessert."
        elif occasion == "superhero":
            return "Buffet with high-energy and healthy food."
        else:
            return "Custom menu for the butler."

    # Alfred, the butler, preparing the menu for the party
    agent = CodeAgent(tools=[suggest_menu], model=InferenceClientModel())

    # Preparing the menu for the party
    agent.run("Prepare a formal menu for the party.")


In [7]:
if "code_agents" in external_exercises:
    from smolagents import CodeAgent, InferenceClientModel
    import numpy as np
    import time
    import datetime

    agent = CodeAgent(tools=[], model=InferenceClientModel(), additional_authorized_imports=['datetime'])

    agent.run(
        """
        Alfred needs to prepare for the party. Here are the tasks:
        1. Prepare the drinks - 30 minutes
        2. Decorate the mansion - 60 minutes
        3. Set up the menu - 45 minutes
        4. Prepare the music and playlist - 45 minutes

        If we start right now, at what time will the party be ready?
        """
    )


In [8]:
if "hub_transfer" in external_exercises:
    # Change to your username and repo name
    agent.push_to_hub('sergiopaniego/AlfredAgent')


In [9]:
if "hub_transfer" in external_exercises:
    # Change to your username and repo name
    alfred_agent = agent.from_hub('sergiopaniego/AlfredAgent', trust_remote_code=True)

    alfred_agent.run("Give me the best playlist for a party at Wayne's mansion. The party idea is a 'villain masquerade' theme")


In [10]:
if "code_agents" in external_exercises:
    from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, InferenceClientModel, Tool, tool, VisitWebpageTool

    @tool
    def suggest_menu(occasion: str) -> str:
        """
        Suggests a menu based on the occasion.
        Args:
            occasion: The type of occasion for the party.
        """
        if occasion == "casual":
            return "Pizza, snacks, and drinks."
        elif occasion == "formal":
            return "3-course dinner with wine and dessert."
        elif occasion == "superhero":
            return "Buffet with high-energy and healthy food."
        else:
            return "Custom menu for the butler."

    @tool
    def catering_service_tool(query: str) -> str:
        """
        This tool returns the highest-rated catering service in Gotham City.

        Args:
            query: A search term for finding catering services.
        """
        # Example list of catering services and their ratings
        services = {
            "Gotham Catering Co.": 4.9,
            "Wayne Manor Catering": 4.8,
            "Gotham City Events": 4.7,
        }

        # Find the highest rated catering service (simulating search query filtering)
        best_service = max(services, key=services.get)

        return best_service

    class SuperheroPartyThemeTool(Tool):
        name = "superhero_party_theme_generator"
        description = """
        This tool suggests creative superhero-themed party ideas based on a category.
        It returns a unique party theme idea."""

        inputs = {
            "category": {
                "type": "string",
                "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
            }
        }

        output_type = "string"

        def forward(self, category: str):
            themes = {
                "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
                "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
                "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
            }

            return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")


    # Alfred, the butler, preparing the menu for the party
    agent = CodeAgent(
        tools=[
            DuckDuckGoSearchTool(),
            VisitWebpageTool(),
            suggest_menu,
            catering_service_tool,
            SuperheroPartyThemeTool(),
    	FinalAnswerTool()
        ],
        model=InferenceClientModel(),
        max_steps=10,
        verbosity_level=2
    )

    agent.run("Give me the best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")


In [11]:
if "langfuse" in external_exercises:
    install_packages(*['opentelemetry-sdk', 'opentelemetry-exporter-otlp', 'openinference-instrumentation-smolagents', 'langfuse'])


In [12]:
if "langfuse" in external_exercises:
    import os
    from google.colab import userdata

    for name in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
        os.environ[name] = userdata.get(name)
    os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")


In [13]:
if "langfuse" in external_exercises:
    from langfuse import get_client

    langfuse = get_client()

    # Verify connection
    if langfuse.auth_check():
        print("Langfuse client is authenticated and ready!")
    else:
        print("Authentication failed. Please check your credentials and host.")


In [14]:
if "langfuse" in external_exercises:
    from openinference.instrumentation.smolagents import SmolagentsInstrumentor

    SmolagentsInstrumentor().instrument()


In [15]:
if "langfuse" in external_exercises:
    from smolagents import CodeAgent, InferenceClientModel

    agent = CodeAgent(tools=[], model=InferenceClientModel())
    alfred_agent = agent.from_hub('sergiopaniego/AlfredAgent', trust_remote_code=True)
    alfred_agent.run("Give me the best playlist for a party at Wayne's mansion. The party idea is a 'villain masquerade' theme")


### Llamadas como código y como JSON

Las dos formas llaman a la misma herramienta local con las mismas consultas.

In [16]:
import json

def web_search(query):
    sample_results = {
        "Best catering services in Gotham City": "Gotham Catering Co. (datos de ejemplo)",
        "Party theme ideas for superheroes": "Baile de máscaras (datos de ejemplo)",
    }
    return sample_results[query]

def dispatch_json_tool_call(serialized_call):
    call = json.loads(serialized_call)
    return {"web_search": web_search}[call["name"]](call["arguments"])

queries = ["Best catering services in Gotham City", "Party theme ideas for superheroes"]
for query in queries:
    direct = web_search(query)
    serialized = json.dumps({"name": "web_search", "arguments": query})
    structured = dispatch_json_tool_call(serialized)
    assert direct == structured
    print(f"{query} → {structured}")


Best catering services in Gotham City → Gotham Catering Co. (datos de ejemplo)
Party theme ideas for superheroes → Baile de máscaras (datos de ejemplo)


In [17]:
if "tool_calling" in external_exercises:
    for query in [
        "Best catering services in Gotham City",
        "Party theme ideas for superheroes"
    ]:
        print(web_search(f"Search for: {query}"))


In [18]:
if "tool_calling" in external_exercises:
    [
        {"name": "web_search", "arguments": "Best catering services in Gotham City"},
        {"name": "web_search", "arguments": "Party theme ideas for superheroes"}
    ]


In [19]:
if "tool_calling" in external_exercises:
    from smolagents import ToolCallingAgent, WebSearchTool, InferenceClientModel

    agent = ToolCallingAgent(tools=[WebSearchTool()], model=InferenceClientModel())

    agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")


```text
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': "best music recommendations for a party at Wayne's         │
│ mansion"}                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
```

### Herramienta de catering

La función de la lección compara las puntuaciones ficticias sin llamar a un proveedor.

In [20]:
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service

best_service = catering_service_tool("catering in Gotham")
assert best_service == "Gotham Catering Co."
print("Mejor puntuación del conjunto de ejemplo:", best_service)


Mejor puntuación del conjunto de ejemplo: Gotham Catering Co.


In [21]:
if "tools" in external_exercises:
    from smolagents import CodeAgent, InferenceClientModel, tool

    # Let's pretend we have a function that fetches the highest-rated catering services.
    @tool
    def catering_service_tool(query: str) -> str:
        """
        This tool returns the highest-rated catering service in Gotham City.

        Args:
            query: A search term for finding catering services.
        """
        # Example list of catering services and their ratings
        services = {
            "Gotham Catering Co.": 4.9,
            "Wayne Manor Catering": 4.8,
            "Gotham City Events": 4.7,
        }

        # Find the highest rated catering service (simulating search query filtering)
        best_service = max(services, key=services.get)

        return best_service


    agent = CodeAgent(tools=[catering_service_tool], model=InferenceClientModel())

    # Run the agent to find the best catering service
    result = agent.run(
        "Can you give me the name of the highest-rated catering service in Gotham City?"
    )

    print(result)   # Output: Gotham Catering Co.


In [22]:
if "tools" in external_exercises:
    from smolagents import Tool, CodeAgent, InferenceClientModel

    class SuperheroPartyThemeTool(Tool):
        name = "superhero_party_theme_generator"
        description = """
        This tool suggests creative superhero-themed party ideas based on a category.
        It returns a unique party theme idea."""

        inputs = {
            "category": {
                "type": "string",
                "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
            }
        }

        output_type = "string"

        def forward(self, category: str):
            themes = {
                "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
                "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
                "futuristic Gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
            }

            return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

    # Instantiate the tool
    party_theme_tool = SuperheroPartyThemeTool()
    agent = CodeAgent(tools=[party_theme_tool], model=InferenceClientModel())

    # Run the agent to generate a party theme idea
    result = agent.run(
        "What would be a good superhero party idea for a 'villain masquerade' theme?"
    )

    print(result)  # Output: "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains."


In [23]:
if "hub_transfer" in external_exercises:
    party_theme_tool.push_to_hub("{your_username}/party_theme_tool", token="<YOUR_HUGGINGFACEHUB_API_TOKEN>")


In [24]:
if "hub_transfer" in external_exercises:
    from smolagents import load_tool, CodeAgent, InferenceClientModel

    image_generation_tool = load_tool(
        "m-ric/text-to-image",
        trust_remote_code=True
    )

    agent = CodeAgent(
        tools=[image_generation_tool],
        model=InferenceClientModel()
    )

    agent.run("Generate an image of a luxurious superhero-themed party at Wayne Manor with made-up superheros.")


In [25]:
if "tools" in external_exercises:
    from smolagents import CodeAgent, InferenceClientModel, Tool

    image_generation_tool = Tool.from_space(
        "black-forest-labs/FLUX.1-schnell",
        name="image_generator",
        description="Generate an image from a prompt"
    )

    model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")

    agent = CodeAgent(tools=[image_generation_tool], model=model)

    agent.run(
        "Improve this prompt, then generate an image of it.",
        additional_args={'user_prompt': 'A grand superhero-themed party at Wayne Manor, with Alfred overseeing a luxurious gala'}
    )


In [26]:
if "serpapi" in external_exercises:
    install_packages(*['-U', 'langchain-community'])


In [27]:
if "serpapi" in external_exercises:
    import os
    from google.colab import userdata

    os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")


In [28]:
if "serpapi" in external_exercises:
    from langchain.agents import load_tools
    from smolagents import CodeAgent, InferenceClientModel, Tool

    search_tool = Tool.from_langchain(load_tools(["serpapi"])[0])

    agent = CodeAgent(tools=[search_tool], model=model)

    agent.run("Search for luxury entertainment ideas for a superhero-themed event, such as live performances and interactive experiences.")


In [29]:
if "mcp" in external_exercises:
    install_packages(*['smolagents[mcp]'])


In [30]:
if "mcp" in external_exercises:
    import os
    from smolagents import ToolCollection, CodeAgent
    from mcp import StdioServerParameters
    from smolagents import InferenceClientModel


    model = InferenceClientModel("Qwen/Qwen2.5-Coder-32B-Instruct")


    server_parameters = StdioServerParameters(
        command="uvx",
        args=["--quiet", "pubmedmcp@0.1.3"],
        env={"UV_PYTHON": "3.12", **os.environ},
    )

    with ToolCollection.from_mcp(server_parameters, trust_remote_code=True) as tool_collection:
        agent = CodeAgent(tools=[*tool_collection.tools], model=model, add_base_tools=True)
        agent.run("Please find a remedy for hangover.")


### Recuperación sobre los documentos de la lección

Usa coincidencia de palabras para practicar la recuperación. El ejemplo de BM25 queda más abajo.

In [31]:
import re

party_ideas = [
    {"text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.", "source": "Party Ideas 1"},
    {"text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.", "source": "Entertainment Ideas"},
    {"text": "For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'", "source": "Catering Ideas"},
    {"text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.", "source": "Decoration Ideas"},
    {"text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.", "source": "Entertainment Ideas"}
]

def terms(text):
    return set(re.findall(r"[A-Za-z]+", text.lower()))

def retrieve_party_ideas(query, limit=3):
    query_terms = terms(query)
    ranked = sorted(party_ideas, key=lambda doc: len(query_terms & terms(doc["text"])), reverse=True)
    return [doc for doc in ranked if query_terms & terms(doc["text"])][:limit]

retrieved = retrieve_party_ideas("catering decorations")
assert retrieved and retrieved[0]["source"] == "Catering Ideas"
assert not retrieve_party_ideas("astronomy")
for doc in retrieved:
    print(f"[{doc['source']}] {doc['text']}")


[Catering Ideas] For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'


In [32]:
if "retrieval" in external_exercises:
    from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

    # Initialize the search tool
    search_tool = DuckDuckGoSearchTool()

    # Initialize the model
    model = InferenceClientModel()

    agent = CodeAgent(
        model=model,
        tools=[search_tool],
    )

    # Example usage
    response = agent.run(
        "Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering."
    )
    print(response)


In [33]:
if "retrieval" in external_exercises:
    from langchain_community.docstore.document import Document
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from smolagents import Tool
    from langchain_community.retrievers import BM25Retriever
    from smolagents import CodeAgent, InferenceClientModel

    class PartyPlanningRetrieverTool(Tool):
        name = "party_planning_retriever"
        description = "Uses semantic search to retrieve relevant party planning ideas for Alfred’s superhero-themed party at Wayne Manor."
        inputs = {
            "query": {
                "type": "string",
                "description": "The query to perform. This should be a query related to party planning or superhero themes.",
            }
        }
        output_type = "string"

        def __init__(self, docs, **kwargs):
            super().__init__(**kwargs)
            self.retriever = BM25Retriever.from_documents(
                docs, k=5  # Retrieve the top 5 documents
            )

        def forward(self, query: str) -> str:
            assert isinstance(query, str), "Your search query must be a string"

            docs = self.retriever.invoke(
                query,
            )
            return "\nRetrieved ideas:\n" + "".join(
                [
                    f"\n\n===== Idea {str(i)} =====\n" + doc.page_content
                    for i, doc in enumerate(docs)
                ]
            )

    # Simulate a knowledge base about party planning
    party_ideas = [
        {"text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.", "source": "Party Ideas 1"},
        {"text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.", "source": "Entertainment Ideas"},
        {"text": "For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'", "source": "Catering Ideas"},
        {"text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.", "source": "Decoration Ideas"},
        {"text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.", "source": "Entertainment Ideas"}
    ]

    source_docs = [
        Document(page_content=doc["text"], metadata={"source": doc["source"]})
        for doc in party_ideas
    ]

    # Split the documents into smaller chunks for more efficient search
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        add_start_index=True,
        strip_whitespace=True,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    docs_processed = text_splitter.split_documents(source_docs)

    # Create the retriever tool
    party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

    # Initialize the agent
    agent = CodeAgent(tools=[party_planning_retriever], model=InferenceClientModel())

    # Example usage
    response = agent.run(
        "Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration options."
    )

    print(response)


### Cálculo de vuelo y paso de datos entre herramientas

La función de distancia del curso se ejecuta con coordenadas fijas. La planificación de ejemplo consume el cálculo.

In [34]:
import math
from typing import Optional, Tuple

def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)

locations = {"Nueva York": (40.7128, -74.0060), "Chicago": (41.8781, -87.6298),
             "Sídney": (-33.8688, 151.2093)}
origin = locations["Chicago"]
options = [(name, calculate_cargo_travel_time(origin, coord)) for name, coord in locations.items()
           if name != "Chicago"]
assert calculate_cargo_travel_time(origin, origin) == 1.0
assert options[0][1] < options[1][1]
for name, hours in sorted(options, key=lambda option: option[1]):
    print(f"Chicago → {name}: {hours:.2f} h")


Chicago → Nueva York: 2.68 h
Chicago → Sídney: 22.82 h


In [35]:
if "map" in external_exercises:
    install_packages(*['smolagents[litellm]', 'plotly', 'geopandas', 'shapely', 'kaleido', '-q'])


In [36]:
if "map" in external_exercises:
    import math
    from typing import Optional, Tuple

    from smolagents import tool


    @tool
    def calculate_cargo_travel_time(
        origin_coords: Tuple[float, float],
        destination_coords: Tuple[float, float],
        cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
    ) -> float:
        """
        Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

        Args:
            origin_coords: Tuple of (latitude, longitude) for the starting point
            destination_coords: Tuple of (latitude, longitude) for the destination
            cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

        Returns:
            float: The estimated travel time in hours

        Example:
            >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
            >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
        """

        def to_radians(degrees: float) -> float:
            return degrees * (math.pi / 180)

        # Extract coordinates
        lat1, lon1 = map(to_radians, origin_coords)
        lat2, lon2 = map(to_radians, destination_coords)

        # Earth's radius in kilometers
        EARTH_RADIUS_KM = 6371.0

        # Calculate great-circle distance using the haversine formula
        dlon = lon2 - lon1
        dlat = lat2 - lat1

        a = (
            math.sin(dlat / 2) ** 2
            + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
        )
        c = 2 * math.asin(math.sqrt(a))
        distance = EARTH_RADIUS_KM * c

        # Add 10% to account for non-direct routes and air traffic controls
        actual_distance = distance * 1.1

        # Calculate flight time
        # Add 1 hour for takeoff and landing procedures
        flight_time = (actual_distance / cruising_speed_kmh) + 1.0

        # Format the results
        return round(flight_time, 2)


    print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))


In [37]:
if "map" in external_exercises:
    import os
    from PIL import Image
    from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool

    model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")


In [38]:
if "map" in external_exercises:
    task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
    Also give me some supercar factories with the same cargo plane transfer time."""


In [39]:
if "map" in external_exercises:
    agent = CodeAgent(
        model=model,
        tools=[GoogleSearchTool("serper"), VisitWebpageTool(), calculate_cargo_travel_time],
        additional_authorized_imports=["pandas"],
        max_steps=20,
    )


In [40]:
if "map" in external_exercises:
    result = agent.run(task)


In [41]:
if "map" in external_exercises:
    result


```python
|  | Location                                             | Travel Time to Gotham (hours) |
|--|------------------------------------------------------|------------------------------|
| 0  | Necropolis Cemetery, Glasgow, Scotland, UK         | 8.60                         |
| 1  | St. George's Hall, Liverpool, England, UK         | 8.81                         |
| 2  | Two Temple Place, London, England, UK             | 9.17                         |
| 3  | Wollaton Hall, Nottingham, England, UK           | 9.00                         |
| 4  | Knebworth House, Knebworth, Hertfordshire, UK    | 9.15                         |
| 5  | Acton Lane Power Station, Acton Lane, Acton, UK  | 9.16                         |
| 6  | Queensboro Bridge, New York City, USA            | 1.01                         |
| 7  | Wall Street, New York City, USA                  | 1.00                         |
| 8  | Mehrangarh Fort, Jodhpur, Rajasthan, India       | 18.34                        |
| 9  | Turda Gorge, Turda, Romania                      | 11.89                        |
| 10 | Chicago, USA                                     | 2.68                         |
| 11 | Hong Kong, China                                 | 19.99                        |
| 12 | Cardington Studios, Northamptonshire, UK        | 9.10                         |
| 13 | Warner Bros. Leavesden Studios, Hertfordshire, UK | 9.13                         |
| 14 | Westwood, Los Angeles, CA, USA                  | 6.79                         |
| 15 | Woking, UK (McLaren)                             | 9.13                         |
```

In [42]:
if "map" in external_exercises:
    agent.planning_interval = 4

    detailed_report = agent.run(f"""
    You're an expert analyst. You make comprehensive reports after visiting many websites.
    Don't hesitate to search for many queries at once in a for loop.
    For each data point that you find, visit the source url to confirm numbers.

    {task}
    """)

    print(detailed_report)


In [43]:
if "map" in external_exercises:
    detailed_report


```python
|  | Location                                         | Travel Time (hours) |
|--|--------------------------------------------------|---------------------|
| 0  | Bridge of Sighs, Glasgow Necropolis, Glasgow, UK | 8.6                 |
| 1  | Wishart Street, Glasgow, Scotland, UK         | 8.6                 |
```

In [44]:
if "map" in external_exercises:
    model = InferenceClientModel(
        "Qwen/Qwen2.5-Coder-32B-Instruct", provider="together", max_tokens=8096
    )

    web_agent = CodeAgent(
        model=model,
        tools=[
            GoogleSearchTool(provider="serper"),
            VisitWebpageTool(),
            calculate_cargo_travel_time,
        ],
        name="web_agent",
        description="Browses the web to find information",
        verbosity_level=0,
        max_steps=10,
    )


In [45]:
if "map" in external_exercises:
    from smolagents.utils import encode_image_base64, make_image_url
    from smolagents import OpenAIServerModel


    def check_reasoning_and_plot(final_answer, agent_memory):
        multimodal_model = OpenAIServerModel("gpt-4o", max_tokens=8096)
        filepath = "saved_map.png"
        assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
        image = Image.open(filepath)
        prompt = (
            f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
            "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
            "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
            "Don't be harsh: if the plot mostly solves the task, it should pass."
            "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
        )
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt,
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": make_image_url(encode_image_base64(image))},
                    },
                ],
            }
        ]
        output = multimodal_model(messages).content
        print("Feedback: ", output)
        if "FAIL" in output:
            raise Exception(output)
        return True


    manager_agent = CodeAgent(
        model=InferenceClientModel("deepseek-ai/DeepSeek-R1", provider="together", max_tokens=8096),
        tools=[calculate_cargo_travel_time],
        managed_agents=[web_agent],
        additional_authorized_imports=[
            "geopandas",
            "plotly",
            "shapely",
            "json",
            "pandas",
            "numpy",
        ],
        planning_interval=5,
        verbosity_level=2,
        final_answer_checks=[check_reasoning_and_plot],
        max_steps=15,
    )


In [46]:
if "map" in external_exercises:
    manager_agent.visualize()


```python
CodeAgent | deepseek-ai/DeepSeek-R1
├── ✅ Authorized imports: ['geopandas', 'plotly', 'shapely', 'json', 'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | Qwen/Qwen2.5-Coder-32B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a google web search for  │ query (`string`): The search      │
            │                             │ your query then returns a string  │ query to perform.                 │
            │                             │ of the top search results.        │ filter_year (`integer`):          │
            │                             │                                   │ Optionally restrict results to a  │
            │                             │                                   │ certain year                      │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
            │                             │ distance.                         │ destination_coords (`array`):     │
            │                             │                                   │ Tuple of (latitude, longitude)    │
            │                             │                                   │ for the destination               │
            │                             │                                   │ cruising_speed_kmh (`number`):    │
            │                             │                                   │ Optional cruising speed in km/h   │
            │                             │                                   │ (defaults to 750 km/h for typical │
            │                             │                                   │ cargo planes)                     │
            │ final_answer                │ Provides a final answer to the    │ answer (`any`): The final answer  │
            │                             │ given problem.                    │ to the problem                    │
            └─────────────────────────────┴───────────────────────────────────┴───────────────────────────────────┘
```

In [47]:
if "map" in external_exercises:
    manager_agent.run("""
    Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
    Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
    Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

    Here's an example of how to plot and return a map:
    import plotly.express as px
    df = px.data.carshare()
    fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
         color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
    fig.show()
    fig.write_image("saved_image.png")
    final_answer(fig)

    Never try to process strings using code: when you have a string to read, just print it and you'll see it.
    """)


In [48]:
if "map" in external_exercises:
    manager_agent.python_executor.state["fig"]


### Entrada de imagen de ejemplo

Verifica la estructura de un mensaje multimodal con un PNG de un píxel. Se construye localmente y no se envía a un modelo.

In [49]:
import base64

sample_png = "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO+jfgAAAABJRU5ErkJggg=="
data_url = f"data:image/png;base64,{sample_png}"
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the image."},
        {"type": "image_url", "image_url": {"url": data_url}},
    ],
}
encoded_image = message["content"][1]["image_url"]["url"].split(",", 1)[1]
assert base64.b64decode(encoded_image, validate=True).startswith(b"\x89PNG\r\n\x1a\n")
print("Mensaje:", message["role"], [part["type"] for part in message["content"]])


Mensaje: user ['text', 'image_url']


In [50]:
if "vision" in external_exercises:
    from PIL import Image
    import requests
    from io import BytesIO

    image_urls = [
        "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg", # Joker image
        "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg" # Joker image
    ]

    images = []
    for url in image_urls:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
        }
        response = requests.get(url,headers=headers)
        image = Image.open(BytesIO(response.content)).convert("RGB")
        images.append(image)


In [51]:
if "vision" in external_exercises:
    from smolagents import CodeAgent, OpenAIServerModel

    model = OpenAIServerModel(model_id="gpt-4o")

    # Instantiate the agent
    agent = CodeAgent(
        tools=[],
        model=model,
        max_steps=20,
        verbosity_level=2
    )

    response = agent.run(
        """
        Describe the costume and makeup that the comic character in these photos is wearing and return the description.
        Tell me if the guest is The Joker or Wonder Woman.
        """,
        images=images
    )


In [52]:
if "vision" in external_exercises:
    {
        'Costume and Makeup - First Image': (
            'Purple coat and a purple silk-like cravat or tie over a mustard-yellow shirt.',
            'White face paint with exaggerated features, dark eyebrows, blue eye makeup, red lips forming a wide smile.'
        ),
        'Costume and Makeup - Second Image': (
            'Dark suit with a flower on the lapel, holding a playing card.',
            'Pale skin, green hair, very red lips with an exaggerated grin.'
        ),
        'Character Identity': 'This character resembles known depictions of The Joker from comic book media.'
    }


In [53]:
if "browser" in external_exercises:
    install_packages(*['smolagents[all]', 'helium', 'selenium', 'python-dotenv'])


In [54]:
if "browser" in external_exercises:
    @tool
    def search_item_ctrl_f(text: str, nth_result: int = 1) -> str:
        """
        Searches for text on the current page via Ctrl + F and jumps to the nth occurrence.
        Args:
            text: The text to search for
            nth_result: Which occurrence to jump to (default: 1)
        """
        elements = driver.find_elements(By.XPATH, f"//*[contains(text(), '{text}')]")
        if nth_result > len(elements):
            raise Exception(f"Match n°{nth_result} not found (only {len(elements)} matches found)")
        result = f"Found {len(elements)} matches for '{text}'."
        elem = elements[nth_result - 1]
        driver.execute_script("arguments[0].scrollIntoView(true);", elem)
        result += f"Focused on element {nth_result} of {len(elements)}"
        return result


    @tool
    def go_back() -> None:
        """Goes back to previous page."""
        driver.back()


    @tool
    def close_popups() -> str:
        """
        Closes any visible modal or pop-up on the page. Use this to dismiss pop-up windows! This does not work on cookie consent banners.
        """
        webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()


In [55]:
if "browser" in external_exercises:
    def save_screenshot(step_log: ActionStep, agent: CodeAgent) -> None:
        sleep(1.0)  # Let JavaScript animations happen before taking the screenshot
        driver = helium.get_driver()
        current_step = step_log.step_number
        if driver is not None:
            for step_logs in agent.logs:  # Remove previous screenshots from logs for lean processing
                if isinstance(step_log, ActionStep) and step_log.step_number <= current_step - 2:
                    step_logs.observations_images = None
            png_bytes = driver.get_screenshot_as_png()
            image = Image.open(BytesIO(png_bytes))
            print(f"Captured a browser screenshot: {image.size} pixels")
            step_log.observations_images = [image.copy()]  # Create a copy to ensure it persists, important!

        # Update observations with current URL
        url_info = f"Current url: {driver.current_url}"
        step_log.observations = url_info if step_logs.observations is None else step_log.observations + "\n" + url_info
        return


In [56]:
if "browser" in external_exercises:
    from smolagents import CodeAgent, OpenAIServerModel, DuckDuckGoSearchTool
    model = OpenAIServerModel(model_id="gpt-4o")

    agent = CodeAgent(
        tools=[DuckDuckGoSearchTool(), go_back, close_popups, search_item_ctrl_f],
        model=model,
        additional_authorized_imports=["helium"],
        step_callbacks=[save_screenshot],
        max_steps=20,
        verbosity_level=2,
    )


In [57]:
if "browser" in external_exercises:
    agent.run("""
    I am Alfred, the butler of Wayne Manor, responsible for verifying the identity of guests at party. A superhero has arrived at the entrance claiming to be Wonder Woman, but I need to confirm if she is who she says she is.

    Please search for images of Wonder Woman and generate a detailed visual description based on those images. Additionally, navigate to Wikipedia to gather key details about her appearance. With this information, I can determine whether to grant her access to the event.
    """ + helium_instructions)


```python
Final answer: Wonder Woman is typically depicted wearing a red and gold bustier, blue shorts or skirt with white stars, a golden tiara, silver bracelets, and a golden Lasso of Truth. She is Princess Diana of Themyscira, known as Diana Prince in the world of men.
```